In [1]:
import pandas as pd
import numpy as np

# Load Austria data with growth variables
df = pd.read_pickle('../data/processed/austria_with_growth.pkl')
print(f"Loaded data: {df.shape}")
print(f"Columns: {len(df.columns)}")
print("Available growth columns:", [col for col in df.columns if 'growth' in col or 'aagr' in col])

Loaded data: (46085, 31)
Columns: 31
Available growth columns: ['growth_2024', 'growth_2023', 'growth_2022', 'aagr_2024']


## Consistent High-Growth Firm Classification (2024)

Following the Belgium notebook logic exactly, with explicit adherence to the recommended rules for handling "n.a." values:

**Recommended Rules Applied:**
- **Rule 1**: If an input employee value needed for a calculation is unavailable, the output variable should also be unavailable.
- **Rule 2**: Do not convert unavailable values into zero growth.
- **Rule 3**: Do not classify firms when the required inputs are unavailable.
- **Rule 4**: For growth-based classification, exclude firms with fewer than 10 employees in the relevant base year.

**Variable**: `ConsistentHighGrowthFirm_2024`

**Classification Rules**:
1. If any required growth variables are unavailable (Rule 1 & 3) → "n.a."
2. If base-year employees (2021) < 10 (Rule 4) → "n.a."
3. Otherwise:
   - Check if yearly growth > 20% in 2022, 2023, 2024
   - Require >20% growth in at least 2 of the 3 years
   - Require AAGR_2024 > 20%
   - If both conditions met → classify as 1 (High-Growth)
   - Otherwise → classify as 0 (Not High-Growth)

**Required Variables**:
- `growth_2022`, `growth_2023`, `growth_2024`
- `aagr_2024`
- `emp_2021_num` (for size threshold)

In [2]:
# Create Consistent High-Growth Firm classification
# Following the recommended rules for handling "n.a." values
# Initialize as object type to hold strings and integers
df['ConsistentHighGrowthFirm_2024'] = pd.Series(['n.a.'] * len(df), dtype=object)

# Rule 3: Check for missing required variables - do not classify when inputs unavailable
required_growth_vars = ['growth_2022', 'growth_2023', 'growth_2024', 'aagr_2024']
missing_mask = (
    (df['growth_2022'] == 'n.a.') |
    (df['growth_2023'] == 'n.a.') |
    (df['growth_2024'] == 'n.a.') |
    (df['aagr_2024'] == 'n.a.')
)

# Rule 4: Check for size threshold (emp_2021 < 10) - exclude small firms
size_mask = df['emp_2021_num'] < 10

# Combined exclusion mask (Rules 3 & 4)
exclusion_mask = missing_mask | size_mask

print(f"Firms excluded due to missing data: {missing_mask.sum()}")
print(f"Firms excluded due to size threshold (<10 employees in 2021): {size_mask.sum()}")
print(f"Total firms excluded: {exclusion_mask.sum()}")
print(f"Firms available for classification: {(~exclusion_mask).sum()}")

# For firms that pass the exclusion criteria, apply classification logic
valid_firms = ~exclusion_mask

if valid_firms.sum() > 0:
    # Convert growth variables to numeric for comparison
    growth_2022_num = pd.to_numeric(df.loc[valid_firms, 'growth_2022'], errors='coerce')
    growth_2023_num = pd.to_numeric(df.loc[valid_firms, 'growth_2023'], errors='coerce')
    growth_2024_num = pd.to_numeric(df.loc[valid_firms, 'growth_2024'], errors='coerce')
    aagr_2024_num = pd.to_numeric(df.loc[valid_firms, 'aagr_2024'], errors='coerce')

    # Check growth > 20% in each year
    growth_2022_high = growth_2022_num > 0.20
    growth_2023_high = growth_2023_num > 0.20
    growth_2024_high = growth_2024_num > 0.20

    # Count years with high growth
    high_growth_years = growth_2022_high.astype(int) + growth_2023_high.astype(int) + growth_2024_high.astype(int)

    # Check AAGR > 20%
    aagr_high = aagr_2024_num > 20

    # Classification: at least 2 years of high growth AND AAGR > 20%
    is_high_growth = (high_growth_years >= 2) & aagr_high

    # Apply classification
    df.loc[valid_firms, 'ConsistentHighGrowthFirm_2024'] = is_high_growth.astype(int)

print("\nClassification Results:")
print(f"High-Growth Firms (1): {(df['ConsistentHighGrowthFirm_2024'] == 1).sum()}")
print(f"Not High-Growth Firms (0): {(df['ConsistentHighGrowthFirm_2024'] == 0).sum()}")
print(f"Unclassified (n.a.): {(df['ConsistentHighGrowthFirm_2024'] == 'n.a.').sum()}")

# Show some examples
print("\nSample classifications:")
sample_df = df[['company_name', 'emp_2021_num', 'growth_2022', 'growth_2023', 'growth_2024', 'aagr_2024', 'ConsistentHighGrowthFirm_2024']].head(10)
print(sample_df.to_string())

Firms excluded due to missing data: 27456
Firms excluded due to size threshold (<10 employees in 2021): 4446
Total firms excluded: 27456
Firms available for classification: 18629

Classification Results:
High-Growth Firms (1): 150
Not High-Growth Firms (0): 18479
Unclassified (n.a.): 27456

Sample classifications:
                                               company_name  emp_2021_num growth_2022 growth_2023 growth_2024 aagr_2024 ConsistentHighGrowthFirm_2024
0                                 LUKOIL INTERNATIONAL GMBH       16200.0   -0.018519        n.a.        n.a.      n.a.                          n.a.
1                                    OMV AKTIENGESELLSCHAFT       22434.0   -0.005616   -0.076923    0.143988  1.641505                             0
2                          OMV GAS MARKETING & TRADING GMBH         148.0    -0.27027         0.0     0.12037 -6.493626                             0
3                                                STRABAG SE       73606.0    0.00182

In [3]:
# Save dataset with classification
output_path = '../data/processed/austria_with_classification.pkl'
df.to_pickle(output_path)
print(f"Dataset with classification saved to: {output_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print("New column added: ConsistentHighGrowthFirm_2024")

Dataset with classification saved to: ../data/processed/austria_with_classification.pkl
Shape: (46085, 32)
Columns: 32
New column added: ConsistentHighGrowthFirm_2024


## Summary

### Consistent High-Growth Firm Classification (2024)

**Variable Created**: `ConsistentHighGrowthFirm_2024`

**Recommended Rules for "n.a." Handling (Explicitly Applied):**
- **Rule 1**: If an input employee value needed for a calculation is unavailable, the output variable should also be unavailable.
- **Rule 2**: Do not convert unavailable values into zero growth.
- **Rule 3**: Do not classify firms when the required inputs are unavailable.
- **Rule 4**: For growth-based classification, exclude firms with fewer than 10 employees in the relevant base year.

**Classification Logic** (following Belgium notebook exactly):
1. **Exclusion Criteria** (Rules 3 & 4):
   - Missing any required growth variables → "n.a."
   - Base year employees < 10 → "n.a."

2. **High-Growth Criteria** (for remaining firms):
   - Year-over-year growth > 20% in at least 2 of 3 years (2022, 2023, 2024)
   - Average Annual Growth Rate > 20% (`aagr_2024 > 20`)
   - Both conditions must be met → classify as 1
   - Otherwise → classify as 0

### Sample Size
- **Total firms**: 46,085
- **Excluded due to missing data**: 27,456 firms
- **Excluded due to size threshold**: 4,446 firms
- **Available for classification**: 18,629 firms

### Classification Results
- **High-Growth Firms (1)**: 150 firms (0.8% of classified firms)
- **Not High-Growth Firms (0)**: 18,479 firms
- **Unclassified ("n.a.")**: 27,456 firms

### Data Output
- **File**: `../data/processed/austria_with_classification.pkl`
- **Shape**: 46,085 rows × 32 columns
- **New column**: `ConsistentHighGrowthFirm_2024`

### Next Steps
This classification follows the Belgium methodology. Check course materials to confirm if additional ESI categories are needed, or if this binary high-growth indicator is sufficient for the assignment.

In [4]:
# Final classification counts
print("Final Classification Results:")
print(f"High-Growth Firms (1): {(df['ConsistentHighGrowthFirm_2024'] == 1).sum()}")
print(f"Not High-Growth Firms (0): {(df['ConsistentHighGrowthFirm_2024'] == 0).sum()}")
print(f"Unclassified ('n.a.'): {(df['ConsistentHighGrowthFirm_2024'] == 'n.a.').sum()}")
print(f"Total: {len(df)}")

high_growth_pct = (df['ConsistentHighGrowthFirm_2024'] == 1).sum() / (df['ConsistentHighGrowthFirm_2024'] != 'n.a.').sum() * 100
print(".1f")

Final Classification Results:
High-Growth Firms (1): 150
Not High-Growth Firms (0): 18479
Unclassified ('n.a.'): 27456
Total: 46085
.1f
